In [1]:
!pip install textblob vaderSentiment scikit-learn tabulate

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels


In [2]:
# Name: NUR IDRAKI HAFIY BI MOHAMAD ZAIDI
# Student ID:is01084550

# 1. IMPORT LIBRARIES
# ==========================================
import pandas as pd
import re
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 2. LOAD DATASET
# ==========================================
df = pd.read_csv("Reviews.csv")

# Select relevant columns only
df = df[['Text', 'Score']]

print("Original Dataset Shape:", df.shape)

# ==========================================
# 3. DATA PREPROCESSING
# ==========================================

# Convert score → sentiment
def label_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['Sentiment'] = df['Score'].apply(label_sentiment)

# Clean text
def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove symbols
    return text

df['Clean_Text'] = df['Text'].astype(str).apply(clean_text)

# Remove missing values
df = df.dropna(subset=['Clean_Text'])

# Reduce dataset size for faster processing
df = df.sample(10000, random_state=42)

print("Processed Dataset Shape:", df.shape)
print(df.head())

# ==========================================
# 4. FEATURE EXTRACTION (TF-IDF)
# ==========================================
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['Clean_Text'])
y = df['Sentiment']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("TF-IDF Features Shape:", X.shape)

# ==========================================
# 5. MACHINE LEARNING MODEL (NAÏVE BAYES)
# ==========================================
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

# ==========================================
# 6. LEXICON-BASED MODELS
# ==========================================
analyzer = SentimentIntensityAnalyzer()

tb_preds = []
vader_preds = []

for text in df['Clean_Text']:
    
    # TextBlob
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0:
        tb_label = 'positive'
    elif polarity < 0:
        tb_label = 'negative'
    else:
        tb_label = 'neutral'
    tb_preds.append(tb_label)

    # VADER
    score = analyzer.polarity_scores(text)['compound']
    if score > 0.05:
        vader_label = 'positive'
    elif score < -0.05:
        vader_label = 'negative'
    else:
        vader_label = 'neutral'
    vader_preds.append(vader_label)

# ==========================================
# 7. MODEL EVALUATION
# ==========================================

print("\n==============================")
print(" NAÏVE BAYES (ML MODEL)")
print("==============================")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

print("\n==============================")
print(" TEXTBLOB")
print("==============================")
print("Accuracy:", accuracy_score(y, tb_preds))
print(classification_report(y, tb_preds))

print("\n==============================")
print(" VADER")
print("==============================")
print("Accuracy:", accuracy_score(y, vader_preds))
print(classification_report(y, vader_preds))

# ==========================================
# 8. SAVE PROCESSED DATA (FOR SUBMISSION)
# ==========================================
df.to_csv("processed_reviews.csv", index=False)

print("\nProcessed dataset saved as 'processed_reviews.csv'")


Original Dataset Shape: (568454, 2)
Processed Dataset Shape: (10000, 4)
                                                     Text  Score Sentiment  \
165256  Having tried a couple of other brands of glute...      5  positive   
231465  My cat loves these treats. If ever I can't fin...      5  positive   
427827  A little less than I expected.  It tends to ha...      3   neutral   
433954  First there was Frosted Mini-Wheats, in origin...      2  negative   
70260   and I want to congratulate the graphic artist ...      5  positive   

                                               Clean_Text  
165256  having tried a couple of other brands of glute...  
231465  my cat loves these treats if ever i cant find ...  
427827  a little less than i expected  it tends to hav...  
433954  first there was frosted miniwheats in original...  
70260   and i want to congratulate the graphic artist ...  
TF-IDF Features Shape: (10000, 5000)

 NAÏVE BAYES (ML MODEL)
Accuracy: 0.7765
              precis

/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start,

              precision    recall  f1-score   support

    negative       0.52      0.39      0.44      1398
     neutral       0.09      0.02      0.03       750
    positive       0.84      0.94      0.89      7852

    accuracy                           0.79     10000
   macro avg       0.48      0.45      0.45     10000
weighted avg       0.74      0.79      0.76     10000


 VADER
Accuracy: 0.8087
              precision    recall  f1-score   support

    negative       0.59      0.41      0.48      1398
     neutral       0.15      0.04      0.07       750
    positive       0.85      0.95      0.90      7852

    accuracy                           0.81     10000
   macro avg       0.53      0.47      0.48     10000
weighted avg       0.76      0.81      0.78     10000


Processed dataset saved as 'processed_reviews.csv'


In [1]:
    print("The sentiment classification models used in this study include lexicon-based methods (TextBlob and VADER)
and a machine learning method (Naïve Bayes). TextBlob is simple and easy to implement, but it is less accurate
when handling neutral sentiments and complex expressions. VADER performs better because it is designed for 
sentiment analysis and can capture intensity and informal language. However, both methods rely on predefined 
rules and dictionaries.

On the other hand, the Naïve Bayes model shows better performance because it learns patterns from the dataset
using TF-IDF features. Although it requires preprocessing and training, it is more suitable for large datasets
such as Amazon reviews. Therefore, Naïve Bayes provides the highest accuracy, while VADER offers a good balance
between simplicity and performance.")

SyntaxError: EOL while scanning string literal (4105589584.py, line 1)